# Heat equation - ODE viewpoint at fixed spatial discretization

In [ ]:
#    APM41012EP course notebook - Chapter 9 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Heat equation - ODE viewpoint at fixed spatial discretization
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
from dataclasses import dataclass
import numpy as np
from scipy.integrate import solve_ivp
from scipy.sparse import diags
from scipy import sparse
from scipy.sparse.linalg import spsolve
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

## Heat equation

We consider the heat equation in 1D:

$$
\left\{
\begin{aligned}
& \partial_t u(x,t) - D \, \partial_{xx} u(x,t) = 0 \;\; \text{for} \; -L < x < L\\
& \partial_{x} u(-L,t) = \partial_{x} u(L,t) = 0,\\
& u(x,t=t_0)  =  \frac{1}{2 \sqrt{\pi D t_0}} \exp \Bigg(-\frac{x^2}{\sqrt{4Dt_0}} \Bigg),
\end{aligned}
\right.
$$

where the initial condition is thus the elementary solution of the heat equation at time $t_0$,
with homogeneous Neumann boundary conditions.

For the simulation we take $L=5$, $t_0=0.01$ and $D=1$. The final time is taken as $t_0+T$ with $T=0.1$. We also take 1001 mesh points in space, a discretization that will remain fixed for this study, in the ODE spirit. One may refer to the other notebook for the analysis in the PDE sense, where the space step also tends to zero.

In [ ]:
#############################################
class heat_model:

    def __init__(self, d, xmin, xmax, nx) :
        self.d = d
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx-1)
        self.oneoverdxdx = 1/(self.dx*self.dx)
        
        doverdxdx = d/(self.dx**2)
        diag = np.repeat(-2, nx)
        sub_diag = np.repeat([1., 2.], [nx-2, 1])
        self.a = doverdxdx * diags([diag, sub_diag, np.flip(sub_diag)], [0, -1, 1])

    def fcn(self, t, y):
        return self.a.dot(y)
    
    def fcn_exact(self, t):
        xmin = self.xmin
        xmax = self.xmax
        d = self.d
        nx = self.nx
        dx = self.dx
        x = np.linspace(xmin, xmax, nx)
        y = (1/(2*np.sqrt(np.pi*t))) * np.exp(-(x*x)/(4.*t))
        return y

## Explicit Euler method

In [ ]:
@dataclass
class ode_result:
    t: np.ndarray 
    y: np.ndarray    
    
#############################################
def forward_euler(tini, tend, nt, yini, fcn):
    
    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    for it, tn  in enumerate(t[:-1]):
        yn = yn + dt*fcn(tn, yn)
        
    return ode_result(t, yn)

#############################################
def heun(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    for it, tn  in enumerate(t[:-1]):
        k1 = fcn(tn, yn)
        k2 = fcn(tn + dt, yn + dt*k1)
        yn = yn + dt/2*(k1+k2)

    return ode_result(t, yn)

#############################################
def backward_euler(tini, tend, nt, yini, a):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    be_mat = sparse.eye(yini.size) - dt*a

    for it, tn  in enumerate(t[:-1]):
        yn = spsolve(be_mat.tocsr(), yn)

    return ode_result(t, yn)

#############################################
def crank_nicolson(tini, tend, nt, yini, a):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    cn_mat = sparse.eye(yini.size) - 0.5*dt*a

    for it, tn  in enumerate(t[:-1]):
        yn = spsolve(cn_mat.tocsr(), yn + 0.5*dt*a.dot(yn))

    return ode_result(t, yn)

#############################################
def plot_sol(d, tini, tend, nt, xmin, xmax, nx, method):
    
    hm = heat_model(d=d, xmin=xmin, xmax=xmax, nx=nx)
    dx = hm.dx
    print('Spatial discretization: dx = ', dx)
    x = np.linspace(xmin, xmax, nx)
    fcn_exact = hm.fcn_exact
    fcn = hm.fcn
    a = hm.a
    
    yini = fcn_exact(tini)
    yexa = fcn_exact(tend)
    
    sol_qexa = solve_ivp(fcn, (tini, tend), yini, method='Radau', t_eval=[tend], rtol=1e-12, atol=1.e-12)
    yqexa = sol_qexa.y[:,0]

    fig = make_subplots(rows=2, cols=1, subplot_titles=("Solution", "Error"), vertical_spacing=0.15)

    fig.add_trace(go.Scattergl(x=x, y=yexa, name=f'exact sol', legendgroup='sol', line_color='rgb(76,114,176)'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=x, y=yqexa, name=f'quasi-exact sol', mode='lines+markers', legendgroup='sol'), row=1, col=1)

    #nt_vn = int(2*(tend-tini)/(dx*dx)) + 1
    #nt = [nt_vn-20, nt_vn, nt_vn+1000, nt_vn+3000, nt_vn+8000]
    for nt_i in nt:
        #print('dt = ', (tend-tini)/(nt_i-1))
        if (method == backward_euler  or method == crank_nicolson):
            sol_num = method(tini, tend, nt_i, yini, a)
        else:
            sol_num = method(tini, tend, nt_i, yini, fcn)
        ysol = sol_num.y
        err_exa  = np.abs(yexa-ysol)
        err_qexa = np.abs(yqexa-ysol)
        err_dx   = np.abs(yqexa-yexa)
        fig.add_trace(go.Scattergl(visible=False, x=x, y=ysol, name=f'numerical sol', mode='lines+markers', legendgroup='sol', line_color='rgb(85,168,104)'), row=1, col=1)
        fig.add_trace(go.Scattergl(visible=False, x=x, y=err_exa, name='err computed from <br>  the exact solution', mode='lines+markers', legendgroup='err', line_color='rgb(76,114,176)'), row=2, col=1)
        fig.add_trace(go.Scattergl(visible=False, x=x, y=err_qexa, name='err computed from <br>  the quasi-exact solution', mode='lines+markers', legendgroup='err', line_color='rgb(221,132,82)'), row=2, col=1)
        fig.add_trace(go.Scattergl(visible=False, x=x, y=err_dx, name='err in dx', mode='lines+markers', legendgroup='err', line_color='rgb(196,78,82)'), row=2, col=1)

    for i in range(2+4,6+4): fig.data[i].visible = True

    steps = []
    for i, nt_i in enumerate(nt):
        args = [{"visible": [(el==0) or (el==1) or (el==4*i+2) or (el==4*i+3) or (el==4*i+4) or (el==4*i+5)  for el in range(len(fig.data))]}]
        #print(args)
        step = dict(method="update", label = f"{nt_i}", args=args)
        steps.append(step)
    sliders = [dict(active=1, currentvalue={"prefix": "nt : "}, steps=steps)]

    legend = dict(x=0.8, bgcolor='rgba(0,0,0,0)', tracegroupgap=280, groupclick='toggleitem')
    fig.update_layout(height=800, sliders=sliders, legend=legend)
    fig.update_yaxes(exponentformat='e')
    fig.show()
    
#############################################    
def plot_order(d, tini, tend, l_nt, xmin, xmax, nx, method):
    
    hm = heat_model(d=d, xmin=xmin, xmax=xmax, nx=nx)
    dx = hm.dx
    print('Spatial discretization: dx = ', dx)
    x = np.linspace(xmin, xmax, nx)
    fcn_exact = hm.fcn_exact
    fcn = hm.fcn
    a = hm.a

    yini = fcn_exact(tini)
    yexa = fcn_exact(tend)

    l_dt = []
    l_err_qexa = [] 

    for nt in l_nt:

        dt = (tend-tini)/(nt-1)
        
        if (method==backward_euler or method==crank_nicolson):
            sol = method(tini, tend, nt, yini, a)
        else:
            sol = method(tini, tend, nt, yini, fcn)
        ysol = sol.y

        tol = 1.e-9
        t = np.linspace(tini, tend, nt)
        sol_qexa = solve_ivp(fcn, (tini, tend), yini, t_eval=t, method="Radau", rtol=tol, atol=1.e-3*tol)
        yqexa = sol_qexa.y[:,-1]

        # error
        err_qexa = np.abs(yqexa-ysol)
        norm_err_qexa = np.linalg.norm(err_qexa) / np.sqrt(nx)
        #print(f"l2 norm of the error for dt = {dt:.3e} : {norm_err_qexa:.10e}")

        l_dt.append(dt)
        l_err_qexa.append(norm_err_qexa)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=l_dt, y=l_err_qexa, name='error'))
    if (method == forward_euler or method == backward_euler): fig.add_trace(go.Scatter(x=l_dt, y=l_dt, name='slope 1'))
    if (method == heun or method==crank_nicolson): fig.add_trace(go.Scatter(x=l_dt, y=np.array(l_dt)**2, name='slope 2'))
    fig.update_xaxes(type='log', exponentformat='e', title='dt')
    fig.update_yaxes(type='log', exponentformat='e', title='error')
    fig.update_layout(height=500)
    fig.show()

### Solution

In [ ]:
d = 1.

tini = 0.01
tend = 0.11

xmin = -5.
xmax = 5.
nx = 1001
dx = (xmax-xmin)/(nx-1)

dt_lim = (dx*dx) / 2
nt_lim = int((tend-tini)/dt_lim) + 1
nt = [nt_lim-20, nt_lim, nt_lim+1000, nt_lim+3000, nt_lim+8000]

plot_sol(d, tini, tend, nt, xmin, xmax, nx, forward_euler)

### Order

In [ ]:
d = 1.

tini = 0.01
tend = 0.11
nt = [2001,  5001, 20001, 200001]

xmin = -5.
xmax = 5.
nx = 1001

plot_order(d, tini, tend, nt, xmin, xmax, nx, forward_euler)

## Heun's method

### Solution

In [ ]:
d = 1.

tini = 0.01
tend = 0.11

xmin = -5.
xmax = 5.
nx = 1001
dx = (xmax-xmin)/(nx-1)

dt_lim = (dx*dx) / 2
nt_lim = int((tend-tini)/dt_lim) + 1
nt = [nt_lim-20, nt_lim, nt_lim+1000, nt_lim+3000, nt_lim+8000]

plot_sol(d, tini, tend, nt, xmin, xmax, nx, heun)

### Order

In [ ]:
d = 1.

tini = 0.01
tend = 0.11
nt = [2001, 5001, 10001, 20001, 50001, 100001]

xmin = -5.
xmax = 5.
nx = 1001

plot_order(d, tini, tend, nt, xmin, xmax, nx, heun)

### Solution

In [ ]:
d = 1.

tini = 0.01
tend = 0.11

xmin = -5.
xmax = 5.
nx = 1001
dx = (xmax-xmin)/(nx-1)

dt_lim = (dx*dx) / 2
nt_lim = int((tend-tini)/dt_lim) + 1
nt = [nt_lim-1800, nt_lim-1500, nt_lim, nt_lim+1000, nt_lim+3000]

plot_sol(d, tini, tend, nt, xmin, xmax, nx, backward_euler)

### Order

In [ ]:
d = 1.

tini = 0.01
tend = 0.11
nt = [2001,  5001, 20001, 200001]

xmin = -5.
xmax = 5.
nx = 1001

plot_order(d, tini, tend, nt, xmin, xmax, nx, backward_euler)

## Crank-Nicolson method

### Solution

In [ ]:
d = 1.

tini = 0.01
tend = 0.11

xmin = -5.
xmax = 5.
nx = 1001
dx = (xmax-xmin)/(nx-1)

dt_lim = (dx*dx) / 2
nt_lim = int((tend-tini)/dt_lim) + 1
nt = [nt_lim-1800, nt_lim-1500, nt_lim, nt_lim+1000, nt_lim+3000]

plot_sol(d, tini, tend, nt, xmin, xmax, nx, crank_nicolson)

### Order

In [ ]:
d = 1.

tini = 0.01
tend = 0.11
nt = [251, 501, 1001, 2001, 5001, 10001, 20001]

xmin = -5.
xmax = 5.
nx = 1001

plot_order(d, tini, tend, nt, xmin, xmax, nx, crank_nicolson)